# CreditLab market-data pull — run inside LSEG Workspace **Codebook**

Pulls four datasets and writes one JSON export for the local CreditLab XVA engine
(`creditlab.xva.lseg.load_lseg_export`):

1. **USD SOFR zero curve** (IPA ZC curve, with an OIS-chain fallback)
2. **Henry Hub (NYMEX NG) futures strip** — settlement prices
3. **Gas vol** — realized from the front-month continuation (override with implied if you have it)
4. **Single-name CDS spread curves** — three strategies: direct RIC-convention probe,
   entity-name search against the CDS instruments view, and a manual override dict

**How to use:** run cells top to bottom, eyeball each cell's printout, then download the
written JSON (File → Download) and drop it into the repo's `data/processed/` directory.

**Licensing:** university LSEG access covers personal academic research. Keep the export in
`data/processed/` (gitignored) — never commit or redistribute it.

If the CDS cell finds candidate RICs but every quote is empty, that is an **entitlement**
gap (single-name CDS pricing not in the university package) — see the note it prints.

In [ ]:
import json
import math
from datetime import date

import lseg.data as ld

ld.open_session()  # Codebook's default session — no config needed here
print("session open")

In [ ]:
# ---- parameters ------------------------------------------------------------
TICKERS = ["OXY", "DVN", "APA", "HAL", "SLB", "KMI", "WMB"]  # names to try for CDS
CDS_TENORS = ["1Y", "2Y", "3Y", "5Y", "7Y", "10Y"]
N_FUTURES = 36                    # NG contracts to keep
IMPLIED_VOL_OVERRIDE = None       # e.g. 0.58 if you read ATM IV off the vol surface app
# If auto-discovery fails for a name, paste its 5Y senior USD CDS RIC here
# (find it in Workspace: search the company, Debt & Credit → CDS, copy the RIC):
CDS_RIC_OVERRIDES = {}            # e.g. {"OXY": "OXY5YUSAX=R"}
ASOF = date.today()
OUT_PATH = f"lseg_export_{ASOF.strftime('%Y%m%d')}.json"

TENOR_YEARS = {"6M": 0.5, "1Y": 1.0, "2Y": 2.0, "3Y": 3.0, "4Y": 4.0,
               "5Y": 5.0, "7Y": 7.0, "10Y": 10.0, "15Y": 15.0, "20Y": 20.0, "30Y": 30.0}
export = {"version": 1, "asof": ASOF.isoformat(),
          "source": "LSEG Workspace Codebook", "notes": []}
print(f"as-of {ASOF} → {OUT_PATH}")

In [ ]:
# ---- 1. USD SOFR zero curve ------------------------------------------------
zeros = []
try:
    from lseg.data.content.ipa.curves import zc_curves
    resp = zc_curves.Definition(
        curve_definition=zc_curves.ZcCurveDefinitions(
            currency="USD", index_name="SOFR", name="USD SOFR Swap ZC Curve"),
        curve_parameters=zc_curves.ZcCurveParameters(valuation_date=ASOF.isoformat()),
    ).get_data()
    df = resp.data.df
    # keep standard pillars; columns are endDate/tenor/discountFactor/zeroRate-ish
    for _, row in df.iterrows():
        t = TENOR_YEARS.get(str(row.get("tenor", "")))
        rate = row.get("ratePercent")
        if t and rate is not None:
            zeros.append([t, float(rate) / 100.0])
    print(f"IPA ZC curve: {len(zeros)} pillars")
except Exception as e:
    print(f"IPA curve failed ({type(e).__name__}: {e}) — falling back to OIS chain")
    try:
        ois = ld.get_data("0#USDSROIS=", ["PRIMACT_1", "MATUR_DATE"])
        for _, row in ois.dropna().iterrows():
            mat = row["MATUR_DATE"]
            t = (mat.date() - ASOF).days / 365.0 if hasattr(mat, "date") else None
            if t and 0.2 <= t <= 30:
                zeros.append([round(t, 2), float(row["PRIMACT_1"]) / 100.0])
        export["notes"].append("zeros: OIS par rates used as zeros (fallback)")
        print(f"OIS chain fallback: {len(zeros)} pillars")
    except Exception as e2:
        print(f"OIS fallback also failed: {e2}")

zeros = sorted(zeros)[:12]
export["zeros"] = zeros
zeros

In [ ]:
# ---- 2. Henry Hub futures strip ---------------------------------------------
gas = {}
try:
    fut = ld.get_data("0#NG:", ["SETTLE", "TRDPRC_1", "EXPIR_DATE"])
    fut = fut.dropna(subset=["EXPIR_DATE"]).head(N_FUTURES)
    forwards = []
    for _, row in fut.iterrows():
        px = row["SETTLE"] if row["SETTLE"] == row["SETTLE"] else row["TRDPRC_1"]
        exp = row["EXPIR_DATE"]
        d = exp.date() if hasattr(exp, "date") else date.fromisoformat(str(exp)[:10])
        if px == px and d > ASOF:
            forwards.append([d.isoformat(), float(px)])
    forwards.sort()
    gas["forwards"] = forwards
    gas["spot"] = forwards[0][1] if forwards else None
    print(f"NG strip: {len(forwards)} contracts, front {gas['spot']}")
except Exception as e:
    print(f"NG chain failed: {type(e).__name__}: {e}")
gas

In [ ]:
# ---- 3. gas vol -------------------------------------------------------------
if IMPLIED_VOL_OVERRIDE:
    gas["implied_vol"] = float(IMPLIED_VOL_OVERRIDE)
    export["notes"].append("vol: manual implied-vol override")
else:
    try:
        hist = ld.get_history("NGc1", fields="TRDPRC_1", interval="daily", count=260)
        px = hist.dropna().iloc[:, 0].astype(float)
        rets = [math.log(b / a) for a, b in zip(px, px[1:]) if a > 0 and b > 0]
        mean = sum(rets) / len(rets)
        sig = math.sqrt(sum((r - mean) ** 2 for r in rets) / (len(rets) - 1)) * math.sqrt(252)
        gas["implied_vol"] = round(sig, 4)
        export["notes"].append("vol: realized from NGc1 (set IMPLIED_VOL_OVERRIDE for true implied)")
        print(f"realized vol (NGc1, 1y): {sig:.1%}")
    except Exception as e:
        print(f"vol pull failed: {e} — set IMPLIED_VOL_OVERRIDE and rerun")
export["gas"] = gas

In [ ]:
# ---- 4. single-name CDS curves ----------------------------------------------
# Strategy A: probe the RIC convention directly — senior USD composites are
#   usually <root><tenor>US<docclause>=R, e.g. OXY5YUSAX=R (AX = XR14 clause).
# Strategy B: search the CDS instruments view by *company name* (CDS are indexed
#   by legal entity, not equity ticker).
# Strategy C: CDS_RIC_OVERRIDES from the parameters cell.

DOC_CLAUSES = ["AX", "BX", "AM", "XR"]  # try XR14 first, then older clauses


def probe_quotes(rics):
    """Fetch spreads for candidate RICs; return {ric: spread_bp} for live ones."""
    try:
        q = ld.get_data(list(rics), ["PRIMACT_1", "MID_SPREAD"])
    except Exception as e:
        print(f"    quote probe failed: {type(e).__name__}: {e}")
        return {}
    live = {}
    for _, row in q.iterrows():
        val = row.get("MID_SPREAD")
        if val is None or val != val:
            val = row.get("PRIMACT_1")
        if val is not None and val == val:
            live[str(row["Instrument"])] = float(val)
    return live


def find_5y_ric(tk):
    # C: explicit override wins
    if tk in CDS_RIC_OVERRIDES:
        return CDS_RIC_OVERRIDES[tk], "override"
    # A: direct convention probe
    candidates = [f"{tk}5YUS{dc}=R" for dc in DOC_CLAUSES]
    live = probe_quotes(candidates)
    for c in candidates:
        if c in live:
            return c, "convention"
    # B: search by company name against the CDS view
    try:
        name_df = ld.get_data([tk], ["TR.CommonName"])
        company = str(name_df.iloc[0, 1]) if len(name_df) else tk
    except Exception:
        company = tk
    try:
        view = getattr(ld.discovery.Views, "CDS_INSTRUMENTS", ld.discovery.Views.SEARCH_ALL)
        hits = ld.discovery.search(query=f"{company} CDS 5Y USD senior",
                                   view=view, select="RIC,DocumentTitle", top=10)
        print(f"    search hits for {company!r}:")
        for _, h in hits.iterrows():
            print(f"      {h.get('RIC', '?'):24s} {str(h.get('DocumentTitle', ''))[:60]}")
        for _, h in hits.iterrows():
            ric = str(h.get("RIC", ""))
            if "5Y" in ric and probe_quotes([ric]):
                return ric, "search"
    except Exception as e:
        print(f"    search failed: {type(e).__name__}: {e}")
    return None, None


cds = {}
dead_candidates = 0
for tk in TICKERS:
    print(f"{tk}:")
    ric5, how = find_5y_ric(tk)
    if not ric5:
        print("    no live 5Y CDS RIC — skipped")
        dead_candidates += 1
        continue
    rics = {t: ric5.replace("5Y", t) for t in CDS_TENORS}
    live = probe_quotes(rics.values())
    spreads = sorted(
        [TENOR_YEARS[t], live[r] / 10_000.0] for t, r in rics.items() if r in live
    )
    if spreads:
        cds[tk] = {"recovery": 0.4, "spreads": spreads}
        print(f"    {len(spreads)} tenors via {how} ({ric5})")
    else:
        print(f"    {ric5} found but tenor quotes empty")

export["cds"] = cds
print(f"\nCDS curves: {sorted(cds)}")
if not cds and dead_candidates == len(TICKERS):
    print("\nNothing returned for ANY name — likely an entitlement gap, not a naming\n"
          "problem. Check in Workspace: search 'Occidental Petroleum', open Debt &\n"
          "Credit → CDS. If you can see spreads there, copy the exact RIC into\n"
          "CDS_RIC_OVERRIDES above and rerun this cell. If the CDS page itself is\n"
          "empty, the university package excludes single-name CDS pricing — use the\n"
          "repo's sample fixture schema and enter spreads manually from another source.")

In [ ]:
# ---- assemble, validate, write ----------------------------------------------
problems = []
if not export.get("zeros"):
    problems.append("zeros missing")
if not export.get("gas", {}).get("forwards"):
    problems.append("NG forwards missing")
if not export.get("gas", {}).get("implied_vol"):
    problems.append("vol missing")
if not export.get("cds"):
    problems.append("no CDS curves (spread-based CVA won't be available)")

if problems:
    print("INCOMPLETE — fix before using locally:", "; ".join(problems))
else:
    print("export complete")

with open(OUT_PATH, "w") as f:
    json.dump(export, f, indent=1)
print(f"wrote {OUT_PATH} — download it (File → Download) into data/processed/")